In [1]:
import pyspark.sql.functions as f
import pyspark.sql.types as t
import pandas as pd

from gentropy.common.session import Session

Loading BokehJS ...

/Users/dc16/gentropy/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [2]:
session = Session(
    extended_spark_conf={
        "spark.driver.memory": "13g",
    }
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/14 11:36:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
qualified_credible_sets = session.spark.read.parquet("/users/dc16/data/gentropy_paper/qualifying_credible_sets")
qualified_measurement_credible_sets = session.spark.read.parquet(
    "/users/dc16/data/gentropy_paper/qualifying_measurement_credible_sets"
)
coloc_df = session.spark.read.parquet("/users/dc16/data/releases/25.06/coloc/")
ecaviar_df = session.spark.read.parquet("/users/dc16/data/releases/25.06/ecaviar/")
trans_pqtl_cs = session.spark.read.parquet("/users/dc16/data/releases/25.06/credible_set/").filter(
    f.col("isTransQTL") == "true"
)

In [4]:
trans_pqtl_cs.count()

17678

In [5]:
credible_sets = (
    qualified_credible_sets.drop("nCases", "nControls").unionByName(qualified_measurement_credible_sets).persist()
)

In [6]:
(
    coloc_df.join(
        credible_sets.select(f.col("studyLocusId").alias("cs_studyLocusId")),
        (f.col("leftStudyLocusId") == f.col("cs_studyLocusId")),
        "semi",
    )
    .unionByName(
        coloc_df.join(
            credible_sets.select(f.col("studyLocusId").alias("cs_studyLocusId")),
            (f.col("rightStudyLocusId") == f.col("cs_studyLocusId")),
            "semi",
        )
    )
    .distinct()
    .write.parquet("/users/dc16/output/gentropy_paper/coloc_temp.parquet", mode="overwrite")
)
(
    ecaviar_df.join(
        credible_sets.select(f.col("studyLocusId").alias("cs_studyLocusId")),
        (f.col("leftStudyLocusId") == f.col("cs_studyLocusId")),
        "semi",
    )
    .unionByName(
        ecaviar_df.join(
            credible_sets.select(f.col("studyLocusId").alias("cs_studyLocusId")),
            (f.col("rightStudyLocusId") == f.col("cs_studyLocusId")),
            "semi",
        )
    )
    .distinct()
    .write.parquet("/users/dc16/output/gentropy_paper/ecaviar_temp.parquet", mode="overwrite")
)

In [7]:
coloc_qcs = session.spark.read.parquet("/users/dc16/output/gentropy_paper/coloc_temp.parquet")
ecaviar_qcs = session.spark.read.parquet("/users/dc16/output/gentropy_paper/ecaviar_temp.parquet")

In [8]:
coloc_count = coloc_qcs.select("leftStudyLocusId", "rightStudyLocusId").count()
ecaviar_count = ecaviar_qcs.select("leftStudyLocusId", "rightStudyLocusId").count()
coloc_h4_count = coloc_qcs.filter(f.col("h4") >= 0.8).count()
ecaviar_clpp_count = ecaviar_qcs.filter(f.col("clpp") >= 0.01).count()

print(f"Total number of coloc overlaps: {coloc_count:,}")
print(f"Total number of ecaviar overlaps: {ecaviar_count:,}")
print(f"Number of coloc overlaps with H4 > 0.8: {coloc_h4_count:,}")
print(f"Number of ecaviar overlaps with clpp > 0.01: {ecaviar_clpp_count:,}")

Total number of coloc overlaps: 31,167,732
Total number of ecaviar overlaps: 61,484,864
Number of coloc overlaps with H4 > 0.8: 24,536,009
Number of ecaviar overlaps with clpp > 0.01: 41,398,927


In [9]:
(
    coloc_qcs.groupBy("rightStudyType")
    .agg(
        f.format_number(f.count("*"), 0).alias("COLOC overlaps"),
        f.format_number(f.count(f.when(f.col("h4") >= 0.8, 1)), 0).alias("H4 > 0.8"),
    )
    .show()
)

+--------------+--------------+----------+
|rightStudyType|COLOC overlaps|  H4 > 0.8|
+--------------+--------------+----------+
|          gwas|    19,378,948|17,527,430|
|          sqtl|     1,006,729|   618,563|
|          pqtl|     1,402,949| 1,286,119|
|         tuqtl|     1,871,805| 1,143,406|
|          eqtl|     6,756,851| 3,742,702|
|        sceqtl|       750,450|   217,789|
+--------------+--------------+----------+



In [10]:
(
    ecaviar_qcs.groupBy("rightStudyType")
    .agg(
        f.format_number(f.count("*"), 0).alias("eCAVIAR overlaps"),
        f.format_number(f.count(f.when(f.col("clpp") >= 0.01, 1)), 0).alias("CLPP > 1%"),
    )
    .show()
)

+--------------+----------------+----------+
|rightStudyType|eCAVIAR overlaps| CLPP > 1%|
+--------------+----------------+----------+
|          gwas|      41,751,740|33,087,583|
|          sqtl|       1,771,258|   710,271|
|          pqtl|       2,158,787| 1,777,217|
|         tuqtl|       3,279,095| 1,334,596|
|          eqtl|      11,380,938| 4,259,718|
|        sceqtl|       1,143,046|   229,542|
+--------------+----------------+----------+



In [11]:
credible_sets.count()

520975

In [35]:
cs_signif_colocalisation = (
    credible_sets.join(
        coloc_df.filter((f.col("h4") >= 0.8) & (f.col("rightStudyType") != "gwas")).unionByName(
            ecaviar_df.filter((f.col("clpp") >= 0.01) & (f.col("rightStudyType") != "gwas")), True
        ),
        f.col("studyLocusId") == f.col("leftStudyLocusId"),
        "semi",
    )
    .distinct()
    .persist()
)
cs_signif_colocalisation.count()

330584

In [37]:
330584 / 520975

0.6345486827582898

In [39]:
coloc_df_no_trans = coloc_df.join(trans_pqtl_cs, f.col("rightStudyLocusId") == f.col("studyLocusId"), "anti")
ecaviar_df_no_trans = ecaviar_df.join(trans_pqtl_cs, f.col("rightStudyLocusId") == f.col("studyLocusId"), "anti")

In [40]:
cs_signif_colocalisation = (
    credible_sets.join(
        coloc_df_no_trans.filter((f.col("h4") >= 0.8) & (f.col("rightStudyType") != "gwas")).unionByName(
            ecaviar_df_no_trans.filter((f.col("clpp") >= 0.01) & (f.col("rightStudyType") != "gwas")), True
        ),
        f.col("studyLocusId") == f.col("leftStudyLocusId"),
        "semi",
    )
    .distinct()
    .persist()
)
cs_signif_colocalisation.count()

25/08/14 12:19:48 WARN CacheManager: Asked to cache already cached data.


302264

In [41]:
302264 / 520975

0.5801890685733481

In [ ]:
coloc_qcs.agg(
    f.max("numberColocalisingVariants"),
    f.median("numberColocalisingVariants"),
    f.mean("numberColocalisingVariants"),
).show()

+-------------------------------+----------------------------------+-------------------------------+
|max(numberColocalisingVariants)|median(numberColocalisingVariants)|avg(numberColocalisingVariants)|
+-------------------------------+----------------------------------+-------------------------------+
|                           2879|                               3.0|               90.4679188399079|
+-------------------------------+----------------------------------+-------------------------------+



In [ ]:
ecaviar_qcs.agg(
    f.max("numberColocalisingVariants"),
    f.median("numberColocalisingVariants"),
    f.mean("numberColocalisingVariants"),
).show()

+-------------------------------+----------------------------------+-------------------------------+
|max(numberColocalisingVariants)|median(numberColocalisingVariants)|avg(numberColocalisingVariants)|
+-------------------------------+----------------------------------+-------------------------------+
|                           3077|                               5.0|              85.24669909003946|
+-------------------------------+----------------------------------+-------------------------------+



In [52]:
target = session.spark.read.parquet("/users/dc16/Downloads/target/")
feature_matrix = session.spark.read.parquet("/users/dc16/data/releases/25.06/l2g_feature_matrix/").join(
    target.select("id", "biotype").filter(f.col("biotype") == "protein_coding"),
    f.col("geneId") == f.col("id"),
    "inner",
)

In [53]:
(
    feature_matrix.join(credible_sets.select("studyLocusId"), ["studyLocusId"], "semi")
    .agg(f.countDistinct("studyLocusId"))
    .show()
)

+----------------------------+
|count(DISTINCT studyLocusId)|
+----------------------------+
|                      513568|
+----------------------------+



In [54]:
print("Numbers with at least one significant colocalisation:")
(
    feature_matrix.join(credible_sets.select("studyLocusId"), ["studyLocusId"], "semi")
    .filter(
        (f.col("eQTlColocClppMaximum") >= 0.01)
        | (f.col("eQTlColocH4Maximum") >= 0.8)
        | (f.col("sQTlColocClppMaximum") >= 0.01)
        | (f.col("sQTlColocH4Maximum") >= 0.8)
        | (f.col("pQTlColocClppMaximum") >= 0.01)
        | (f.col("pQTlColocH4Maximum") >= 0.8)
    )
    .agg(
        f.count_distinct("studyLocusId").alias("numberOfCredibleSets"),
        f.count_distinct("geneId").alias("numberOfGenes"),
    )
    .show()
)

Numbers with at least one significant colocalisation:


+--------------------+-------------+
|numberOfCredibleSets|numberOfGenes|
+--------------------+-------------+
|              285229|        14026|
+--------------------+-------------+



In [ ]:
print("Percentage of qualified credible sets with at least one significant protein-coding mol-QTL colocalisation:")
(285229 / 520975) * 100

Percentage of qualified credible sets with at least one significant protein-coding mol-QTL colocalisation:


54.74907625125965

25/08/14 17:02:27 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 959494 ms exceeds timeout 120000 ms
25/08/14 17:02:27 WARN SparkContext: Killing executors is not supported by current scheduler.
25/08/14 17:18:28 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [ ]:
print("Numbers with a significant eQTL in feature matrix:")
(
    feature_matrix.join(credible_sets.select("studyLocusId"), ["studyLocusId"], "semi")
    .filter((f.col("eQtlColocClppMaximum") >= 0.01) | (f.col("eQtlColocH4Maximum") >= 0.8))
    .agg(
        f.count_distinct("studyLocusId").alias("numberOfCredibleSets"),
        f.count_distinct("geneId").alias("numberOfGenes"),
    )
    .show()
)

Numbers with a significant eQTL in feature matrix:


+--------------------+-------------+
|numberOfCredibleSets|numberOfGenes|
+--------------------+-------------+
|              253643|        13395|
+--------------------+-------------+



In [ ]:
print("Numbers with a significant pQTL in feature matrix:")
(
    feature_matrix.join(credible_sets.select("studyLocusId"), ["studyLocusId"], "semi")
    .filter((f.col("pQtlColocClppMaximum") >= 0.01) | (f.col("pQtlColocH4Maximum") >= 0.8))
    .agg(
        f.count_distinct("studyLocusId").alias("numberOfCredibleSets"),
        f.count_distinct("geneId").alias("numberOfGenes"),
    )
    .show()
)

Numbers with a significant pQTL in feature matrix:


+--------------------+-------------+
|numberOfCredibleSets|numberOfGenes|
+--------------------+-------------+
|               43038|         1522|
+--------------------+-------------+



In [ ]:
print("Numbers with a significant sQTL in feature matrix:")
(
    feature_matrix.join(credible_sets.select("studyLocusId"), ["studyLocusId"], "semi")
    .filter((f.col("sQtlColocClppMaximum") >= 0.01) | (f.col("sQtlColocH4Maximum") >= 0.8))
    .agg(
        f.count_distinct("studyLocusId").alias("numberOfCredibleSets"),
        f.count_distinct("geneId").alias("numberOfGenes"),
    )
    .show()
)

Numbers with a significant sQTL in feature matrix:


+--------------------+-------------+
|numberOfCredibleSets|numberOfGenes|
+--------------------+-------------+
|              167703|         9406|
+--------------------+-------------+



In [ ]:
coloc_sceqtl = coloc_qcs.filter(f.col("rightStudyType") == "sceqtl")
ecaviar_sceqtl = ecaviar_qcs.filter(f.col("rightStudyType") == "sceqtl")
coloc_wo_sceqtl = coloc_qcs.filter(f.col("rightStudyType") != "sceqtl")
ecaviar_wo_sceqtl = ecaviar_qcs.filter(f.col("rightStudyType") != "sceqtl")

coloc_sceqtl.write.parquet("gs://genetics-portal-dev-analysis/dc16/output/coloc_sceqtl", mode="overwrite")
ecaviar_sceqtl.write.parquet("gs://genetics-portal-dev-analysis/dc16/output/ecaviar_sceqtl", mode="overwrite")
coloc_wo_sceqtl.write.parquet("gs://genetics-portal-dev-analysis/dc16/output/coloc_wo_sceqtl", mode="overwrite")
ecaviar_wo_sceqtl.write.parquet("gs://genetics-portal-dev-analysis/dc16/output/ecaviar_wo_sceqtl", mode="overwrite")

25/08/14 10:23:55 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-google-hadoop-file-system.properties,hadoop-metrics2.properties


Py4JJavaError: An error occurred while calling o379.parquet.
: java.io.IOException: Error accessing gs://genetics-portal-dev-analysis/dc16/output/coloc_sceqtl
	at com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageImpl.getObject(GoogleCloudStorageImpl.java:2346)
	at com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageImpl.getItemInfo(GoogleCloudStorageImpl.java:2235)
	at com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageFileSystemImpl.getFileInfoInternal(GoogleCloudStorageFileSystemImpl.java:1087)
	at com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageFileSystemImpl.getFileInfoInternal(GoogleCloudStorageFileSystemImpl.java:1058)
	at com.google.cloud.hadoop.repackaged.gcs.com.google.cloud.hadoop.gcsio.GoogleCloudStorageFileSystemImpl.getFileInfo(GoogleCloudStorageFileSystemImpl.java:1026)
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem